# Phase 2: Identity Regressor Demographic Fine-Tuning (Stage 1)

**Objective:** Fine-tune MICA's 512-to-300 mapping network on target demographic scans using:
- 2× NVIDIA Tesla T4 GPUs via PyTorch DDP (`torchrun --nproc_per_node=2`).
- Mixed Precision training via modern PyTorch AMP (`torch.amp.autocast('cuda', dtype=torch.float16)`).
- Combined L1 shape loss + 3D FLAME vertex reconstruction loss ($< 0.90\text{mm}$ error target).
- 11.5-hour wall-clock safeguard saving emergency checkpoints before Kaggle's 12-hour session cutoff.

In [ ]:
# ── CELL 1: Environment Check & Safe Kernel Restart ─────────────────────────────
import sys
import subprocess
import os
import numpy as np

print(f"[Setup] Active NumPy version: {np.__version__}")
if np.__version__.startswith("2."):
    print("Detected NumPy 2.x. Downgrading to 1.26.4 for C-extension ABI stability...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy==1.26.4", "--quiet"])
    print("Restarting kernel...")
    os.kill(os.getpid(), 9)
else:
    print("NumPy is compatible. Proceeding.")

In [ ]:
# ── CELL 2: Install Repository Dependencies & GPU Tools ─────────────────────────
!pip install -r requirements.txt --quiet

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA GPU accelerator required for fine-tuning!"
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPU(s):")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ── CELL 3: Dataset & FLAME Model Asset Discovery ───────────────────────────────
from pathlib import Path

# Look for attached Kaggle dataset inputs or local fallback
DATA_DIR = Path('/kaggle/input/face-geo-training-data-stage1')
if not DATA_DIR.exists():
    DATA_DIR = Path('./data/training_stage1')

FLAME_PATH = Path('/kaggle/input/flame-model/generic_model.pkl')
if not FLAME_PATH.exists():
    FLAME_PATH = Path('./data/flame_model/generic_model.pkl')

PRETRAINED_MICA = Path('/kaggle/input/mica-pretrained/mica.tar')
if not PRETRAINED_MICA.exists():
    PRETRAINED_MICA = Path('./models_cache/mica/pretrained.tar')

print(f"Training Data Path: {DATA_DIR.resolve()}")
print(f"FLAME Model Path:   {FLAME_PATH.resolve()} (Exists: {FLAME_PATH.exists()})")
print(f"Pretrained MICA:    {PRETRAINED_MICA.resolve()} (Exists: {PRETRAINED_MICA.exists()})")

In [ ]:
# ── CELL 4: Launch Multi-GPU DDP Training ──────────────────────────────────────
import torch
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
!torchrun --nproc_per_node={n_gpus} src/stage1_identity/trainer.py \
    --data_dir {DATA_DIR} \
    --flame_path {FLAME_PATH} \
    --checkpoint_dir checkpoints/stage1_identity \
    --pretrained {PRETRAINED_MICA} \
    --lr 1e-4 \
    --batch_size 16 \
    --epochs 50 \
    --max_hours 11.5

In [ ]:
# ── CELL 5: Validation Evaluation on NoW Benchmark Metric ───────────────────────
import torch
from pathlib import Path
from src.stage1_identity.inference import MappingNetwork
from src.stage1_identity.data import MICAIdentityDataset
from src.utils.flame_model import FLAMEModel
import numpy as np

ckpt_best = Path('checkpoints/stage1_identity/regressor_best.pt')
if ckpt_best.exists() and FLAME_PATH.exists() and DATA_DIR.exists():
    flame = FLAMEModel(str(FLAME_PATH))
    shapedirs = torch.from_numpy(flame.shapedirs).float().cuda()
    
    model = MappingNetwork(512, 300, 300, hidden=3).cuda()
    model.load_state_dict(torch.load(ckpt_best))
    model.eval()
    
    val_ds = MICAIdentityDataset(str(DATA_DIR), split='val', val_ratio=0.1)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False)
    
    errors_mm = []
    with torch.no_grad():
        for batch in val_loader:
            feats = batch['feature'].cuda()
            beta_gt = batch['beta_gt'].cuda()
            pred_beta = model(feats)
            diff = pred_beta - beta_gt
            v_diff = torch.einsum('bk,vck->bvc', diff, shapedirs) * 1000.0
            err_per_sample = torch.mean(torch.norm(v_diff, dim=-1), dim=-1) # Mean mm error per head
            errors_mm.extend(err_per_sample.cpu().numpy().tolist())
            
    median_err = float(np.median(errors_mm))
    mean_err = float(np.mean(errors_mm))
    print(f"\n--- Validation Benchmark Results ---")
    print(f"Validation Samples: {len(errors_mm)}")
    print(f"Mean 3D Vertex Error:   {mean_err:.3f} mm")
    print(f"Median 3D Vertex Error: {median_err:.3f} mm (Target: < 0.900 mm)")
    if median_err < 0.90:
        print("SUCCESS: Model outperforms MICA pretrained baseline on target distribution!")
else:
    print("Checkpoint or evaluation dataset not found for local run.")

In [ ]:
# ── CELL 6: Checkpoint Upload to Kaggle Models Registry ────────────────────────
!python scripts/upload_to_kaggle_models.py \
    --checkpoint_dir checkpoints/stage1_identity \
    --handle your_username/face-geo-stage1-identity/pytorch/v1 \
    --version_notes "Stage 1 MICA fine-tuned on demographic scans with vertex L1 loss" \
    --stage 1